# NuNo vs RPT — Colab G4 paper-config comparison
Cấu hình Qwen bám Table 5 của paper: **14B → 1.5B**, global batch 64, LR 5e-6, cosine + 10% warmup, length 1024, 2 epochs, λ=0.2, K=128, d′=256, 4 layers trong [0.20, 0.85].

> Colab G4 dùng RTX PRO 6000 Blackwell 96 GB, nên notebook chạy teacher 14B FP16 nguyên bản (`TEACHER_4BIT=False`). Nhánh 4-bit chỉ tự bật nếu notebook bị chuyển sang GPU dưới 35 GB.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(gpu_name, round(gpu_gb, 1), 'GB')

In [ ]:
%pip install -q transformers==4.57.3 peft==0.18.1 datasets deepspeed bitsandbytes accelerate rouge-score numerize rich wandb celery nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess
REPO = Path('/content/nuno-kd')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/chiiipk/nuno-kd.git', str(REPO)], check=True)
os.chdir(REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run(['python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)

## Data
Đặt `generated_train.jsonl` Qwen hiện có vào Google Drive. Cell tự tìm dưới `/content/drive/MyDrive/NuNo/`. File phải có 79.751 dòng và hai key `prompt`, `generated_text`.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/NuNo')
candidates = list(DATA_ROOT.rglob('generated_train.jsonl'))
qwen = [p for p in candidates if 'Qwen' in str(p) or 'qwen' in str(p)]
RAW_JSONL = (qwen or candidates or [None])[0]
assert RAW_JSONL and RAW_JSONL.exists(), 'Upload file Qwen generated_train.jsonl vào MyDrive/NuNo'
with RAW_JSONL.open() as f:
    first = json.loads(next(f))
assert {'prompt', 'generated_text'} <= first.keys()
print('Data:', RAW_JSONL, '| MB:', round(RAW_JSONL.stat().st_size/2**20, 1))

In [ ]:
# ===== Paper configuration (Qwen column, Table 5) =====
STUDENT = 'Qwen/Qwen2.5-1.5B-Instruct'
TEACHER = 'Qwen/Qwen2.5-14B-Instruct'
GLOBAL_BATCH = 64
MICRO_BATCH = 1
GRAD_ACC = GLOBAL_BATCH // MICRO_BATCH  # Colab: one GPU
LR = '5e-6'
EPOCHS = 2
MAX_LENGTH = 1024
WARMUP_RATIO = '0.1'
NNM_RATIO = '0.2'
K_CENTROIDS = '128'
D_PRIME = '256'
CENTROID_BATCHES = '500'
N_LAYERS = '4'
TEACHER_4BIT = gpu_gb < 35  # False trên G4 RTX PRO 6000 96 GB
SEED = 10
print('Teacher 4-bit:', TEACHER_4BIT, '| effective batch:', MICRO_BATCH * GRAD_ACC)

In [ ]:
PROCESSED_ROOT = Path('/content/processed_data')
DATA_DIR = PROCESSED_ROOT / TEACHER
if not (DATA_DIR / 'train_0.idx').exists():
    subprocess.run([
        'python3', 'tools/process_data_ultraInteract.py', '--data-dir', str(RAW_JSONL),
        '--processed-data-dir', str(PROCESSED_ROOT), '--model-path', TEACHER,
        '--data-process-workers', '2', '--max-prompt-length', '512',
        '--max-length', str(MAX_LENGTH), '--dev-num', '200', '--only-prompt',
        '--model-type', 'qwen'
    ], check=True, env={**os.environ, 'PYTHONPATH': '.'})
assert (DATA_DIR / 'train_0.idx').exists() and (DATA_DIR / 'valid_0.idx').exists()
print('Processed:', DATA_DIR)

In [ ]:
def run_paper_config(variant: str, train_num: int = -1):
    assert variant in {'nuno', 'rpt'}
    output = DATA_ROOT / 'results' / f'{variant}_paper_qwen14b_to_1p5b'
    cmd = [
      'torchrun', '--standalone', '--nproc_per_node=1', 'finetune.py',
      '--base-path', '.', '--model-path', STUDENT, '--teacher-model-path', TEACHER,
      '--ckpt-name', 'qwen2.5-1.5B-it', '--teacher-ckpt-name', 'qwen2.5-14B-it',
      '--teacher-model-fp16', '--n-gpu', '1', '--data-dir', str(DATA_DIR),
      '--num-workers', '2', '--train-num', str(train_num), '--dev-num', '-1',
      '--lr', LR, '--lr-min', '0', '--batch-size', str(MICRO_BATCH),
      '--eval-batch-size', '1', '--gradient-accumulation-steps', str(GRAD_ACC),
      '--gradient-checkpointing', '--warmup-ratio', WARMUP_RATIO,
      '--lr-decay-style', 'cosine', '--weight-decay', '1e-2', '--clip-grad', '1.0',
      '--epochs', str(EPOCHS), '--kd-ratio', '1.0', '--temperature', '1.0',
      '--max-length', str(MAX_LENGTH), '--max-prompt-length', '512', '--do-train',
      '--save-interval', '-1', '--eval-interval', '-1', '--log-interval', '10',
      '--mid-log-num', '-1', '--save', str(output), '--seed', str(SEED),
      '--deepspeed', '--deepspeed_config', 'configs/deepspeed/ds_config_zero2_offload.json',
      '--type', 'adaptive-sfkl', '--skew-alpha', '0.1', '--do-sample', '--top-k', '0',
      '--top-p', '1.0', '--student-gen', '--gen-num-beams', '1', '--gen-top-p', '1.0',
      '--init-threshold', '0.0', '--loss-eps', '0.1', '--capacity', '1000',
      '--replay-ratio', 'decreasing', '--mixed-alpha', '0.5', '--nnm',
      '--loss-variant', variant, '--nnm-ratio', NNM_RATIO, '--nnm-K', K_CENTROIDS,
      '--nnm-n-layers', N_LAYERS, '--nnm-d-prime', D_PRIME,
      '--nnm-centroid-batches', CENTROID_BATCHES, '--nnm-eta', '0.05',
      '--nnm-T-dead', '50', '--nnm-ns-iters', '5', '--delta-threshold', '0.03'
    ]
    if TEACHER_4BIT:
        cmd.append('--teacher-load-in-4bit')
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_DISABLED': 'true',
           'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    print('Running', variant, '->', output)
    subprocess.run(cmd, check=True, env=env)
    return output

## Chạy
Smoke test 64 mẫu trước. Sau khi pass, restart runtime để giải phóng VRAM rồi chạy từng full experiment. Không chạy NuNo và RPT đồng thời.

In [ ]:
run_paper_config('rpt', train_num=64)

In [ ]:
# Full run — bỏ comment đúng một dòng mỗi runtime:
# run_paper_config('nuno', train_num=-1)
# run_paper_config('rpt', train_num=-1)